In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms,models,datasets
from torch.utils.data import DataLoader,random_split
from sklearn.metrics import confusion_matrix,precision_score,recall_score
transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])
dataset=datasets.ImageFolder("dataset",transform=transform)
train_size=int(0.8*len(dataset))
val_size=len(dataset)-train_size
train_set,val_set=random_split(dataset,[train_size,val_size])
train_loader=DataLoader(train_set,batch_size=2,shuffle=True)
val_loader=DataLoader(val_set,batch_size=2)
model=models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc=nn.Linear(model.fc.in_features,2)
for param in model.parameters():
    param.requires_grad=False
for param in model.fc.parameters():
    param.requires_grad=True
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.fc.parameters(),lr=1e-3)
for epoch in range(20):
    model.train()
    for X,Y in train_loader:
        output=model(X)
        loss=criterion(output,Y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
model.eval()
all_preds=[]
all_labels=[]
total=0
total_loss=0
correct=0
with torch.no_grad():
    for X,Y in val_loader:
        output=model(X)
        loss=criterion(output,Y)
        total_loss+=loss.item()*X.size(0)
        pred=torch.argmax(output,dim=1)
        correct+=(pred == Y).sum().item()
        total+=Y.size(0)
        all_preds.extend(pred.tolist())
        all_labels.extend(Y.tolist())
avg_loss=total_loss/len(val_set)
acc=correct/total
cm=confusion_matrix(all_labels,all_preds)
precision=precision_score(all_labels,all_preds,average="binary",zero_division=0)#誤判 tp/(tp+fp)
recall=recall_score(all_labels,all_preds,average="binary",zero_division=0)      #漏判 tp/(tp+fn)
print(f"loss:{loss:.4f}")
print(f"Accuracy:{acc:.4f}")
print(f"Confusion Matrix:{cm}")
print(f"Precision:{precision:.4f}")                #[TN,FP]
print(f"Recall:{recall:04f}")                      #[FN,TP] 正類是1
print("all labels",all_labels)
print("all preds",all_preds)

loss:1.3472
Accuracy:0.5000
Confusion Matrix:[[0 0]
 [1 1]]
Precision:1.0000
Recall:0.500000
all labels [1, 1]
all preds [0, 1]
